In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torchtune.modules import RotaryPositionalEmbeddings


class BKTConfig:
    def __init__(self, n_skills, n_embd=128, n_layer=3, n_head=4, block_size=512, bkt=True):
        self.n_skills = n_skills
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.block_size = block_size
        self.bkt = bkt
        self.dropout = 0.1
        self.bias = False


class SwiGLU(nn.Module):
    def __init__(self, in_dim, hidden_dim, bias=True, dropout=0.0):
        super().__init__()
        self.gate_up_proj = nn.Linear(in_dim, 2 * hidden_dim, bias=bias)
        self.down_proj = nn.Linear(hidden_dim, in_dim, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        gate, up = self.gate_up_proj(x).chunk(2, dim=-1)
        x = F.silu(gate) * up
        x = self.dropout(x)
        return self.down_proj(x)


class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head

        self.q_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.k_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.v_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.resid_dropout = nn.Dropout(config.dropout)

        self.rope = RotaryPositionalEmbeddings(dim=self.head_dim, max_seq_len=config.block_size)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        q, k = self.rope(q, k)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        return y


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = Attention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        hidden_dim = int(8/3 * config.n_embd)
        self.mlp = SwiGLU(config.n_embd, hidden_dim, bias=config.bias, dropout=config.dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class NeuralBKT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.n_skills = config.n_skills
        self.skill_emb = nn.Embedding(config.n_skills, config.n_embd)
        self.correct_emb = nn.Embedding(2, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*[TransformerBlock(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, 4)
        self.skill_params = nn.Linear(config.n_embd, 5)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, obs, return_latent=False, return_all=False, output=None, prior=None, lambd=[50, 50, 50, 1]):
        B, T, D = obs.shape
        assert D == 2, "Expected obs shape (B, T, 2)"
        skill_ids = obs[..., 1].long()
        corrects = obs[..., 0].long().clamp(0, 1)
        token_embeddings = self.skill_emb(skill_ids) + self.correct_emb(corrects)
        x = self.drop(token_embeddings)
        x = self.blocks(x)
        x = self.ln_f(x)
        logit_diff = self.head(x)
        loss = 0.0

        if self.config.bkt:
            skill_indices = torch.arange(self.n_skills, device=obs.device)
            base_logits = self.skill_params(self.skill_emb(skill_indices))
            base_oparams = torch.sigmoid(base_logits[:, 1:])
            base_prior_logits = base_logits[:, 0]
            base_params_expanded = base_oparams[skill_ids]
            adjusted_params = torch.sigmoid(base_params_expanded + logit_diff)
            l, _, g, s = adjusted_params[..., 0], adjusted_params[..., 1], adjusted_params[..., 2], adjusted_params[..., 3]

            constraint_violation = F.relu(l - (1 - s) / (g + 1e-6))
            loss += lambd[0] * constraint_violation.mean()
            loss += lambd[1] * F.relu(g - 0.3).mean()
            loss += lambd[2] * F.relu(s - 0.3).mean()
            loss += lambd[3] * torch.mean(logit_diff ** 2)
            if T > 1:
                loss += lambd[3] * torch.mean((logit_diff[:, 1:] - logit_diff[:, :-1]) ** 2)

            if prior is not None:
                latent = prior.detach()
            else:
                latent = torch.sigmoid(base_prior_logits).repeat(B, 1)

            latents = []
            corrects_pred = torch.zeros(B, T, device=obs.device)

            for t in range(T):
                latent = torch.clamp(latent, min=1e-4, max=1 - 1e-4)
                latents.append(latent.clone())
                l_t = l[:, t]
                g_t = g[:, t]
                s_t = s[:, t]
                skill_t = skill_ids[:, t]
                latent_skill = latent.gather(1, skill_t.unsqueeze(1)).squeeze(1)
                correct_pred = latent_skill * (1 - s_t) + (1 - latent_skill) * g_t
                corrects_pred[:, t] = correct_pred
                true_correct = obs[:, t, 0]

                numerator1 = latent_skill * (1 - s_t)
                denominator1 = numerator1 + (1 - latent_skill) * g_t
                k_t1 = numerator1 / (denominator1 + 1e-8)

                numerator0 = latent_skill * s_t
                denominator0 = numerator0 + (1 - latent_skill) * (1 - g_t)
                k_t0 = numerator0 / (denominator0 + 1e-8)

                k_t = latent.clone()
                mask_correct = true_correct > 0.5
                k_t[mask_correct, skill_t[mask_correct]] = k_t1[mask_correct]
                k_t[~mask_correct, skill_t[~mask_correct]] = k_t0[~mask_correct]

                k_t_skill = k_t.gather(1, skill_t.unsqueeze(1)).squeeze(1)
                #k_t_skill = k_t_skill + (1 - k_t_skill) * l_t

                # Apply learning ONLY if correct
                k_t_skill = torch.where(
                    true_correct > 0.5,
                    k_t_skill + (1 - k_t_skill) * l_t,
                    k_t_skill
                )
    
                k_t.scatter_(1, skill_t.unsqueeze(1), k_t_skill.unsqueeze(1))
                latent = torch.clamp(k_t, 1e-4, 1 - 1e-4)

        if output is not None:
            mask = output[..., 0] != -1
            if mask.any():
                loss += F.binary_cross_entropy(corrects_pred[mask], obs[mask, 0])

        if return_latent:
            return corrects_pred, latents, loss
        elif return_all:
            return corrects_pred, latents, adjusted_params, loss
        return corrects_pred, loss

ModuleNotFoundError: No module named 'torchtune'